# Split data
Split the data into train, val and test datasets. The data is distributed per corrosion inhibitor to determine the capability of the unet to predict different corrosion inhibitors

In [1]:
from pathlib import Path
import os
import random
import shutil

from ordered_set import OrderedSet
import pandas as pd

In [2]:
num_val = 25
num_test = 25

original_path = Path("../rawdata/unet")
inhibitor_list_file = Path("../rawdata/processed/volume-loss/volume-loss.csv")
save_path = Path("../../data/")
before_path = "before"
after_path = "after"
image_path = "image"
height_path = "height"

In [3]:
# Load the list of inhibitor names
inhibitors = pd.read_csv(inhibitor_list_file)
# Filter for all NaCl samples
nacl_samples = inhibitors[inhibitors["name"].str.contains("NaCl", case=False)].id
# Get subids of NaCl samples
nacl_subid = OrderedSet([i[:-1] for i in nacl_samples])
len(nacl_subid)

52

In [4]:
# Get the subids of the folder
total_sub_ids = OrderedSet([i[:-5] for i in os.listdir(original_path / before_path / image_path)])
# Get all NaCl samples contained in the data
contained_nacl_subid = nacl_subid - (nacl_subid - total_sub_ids)
# Remove the NaCl samples
sub_ids = list(total_sub_ids - contained_nacl_subid)
# Shuffle them
random.Random(5).shuffle(sub_ids)
# split data into train, val and test
test_val_ids = sub_ids[:(num_val + num_test)]
train_ids = sub_ids[(num_val + num_test):]
val_ids = test_val_ids[:num_val]
test_ids = test_val_ids[num_val:]
# Add NaCl samples back to training data
train_ids += list(contained_nacl_subid)
len(train_ids), len(val_ids), len(test_ids), len(contained_nacl_subid)

(161, 25, 25, 36)

In [5]:
# Test ist all "problematic" data is in the train set
train_ids.index("3-8-3-"), train_ids.index("3-8-6-"), train_ids.index("4-6-10-"), train_ids.index("2-5-10-"), train_ids.index("2-5-11-")

(2, 149, 112, 17, 119)

In [6]:
def copy_files(id_list, target):
    # Copy images and heights into respective train, val or test folders
    for time in [before_path, after_path]:
        for data_type in [image_path, height_path]:
            source = original_path / time / data_type
            # Remove already existing folder
            try:
                shutil.rmtree(save_path / target / time / data_type)
            except FileNotFoundError:
                pass
            # Create file folder
            os.makedirs(save_path / target / time / data_type)
            for id in id_list:
                for file in source.rglob(f"{id}*"):
                    # Copy files
                    shutil.copy(file, save_path / target / time / data_type)

    # Remove already existing mask folder
    shutil.rmtree(save_path / target / "mask")
    # Create mask folder
    os.makedirs(save_path / target / "mask")
    # Copy masks into respective train, val or test folders
    for id in id_list:
        source = original_path / "mask"
        for file in source.rglob(f"{id}*"):
            # Copy files
            shutil.copy(file, save_path / target / "mask")


In [7]:
# Split the data
copy_files(val_ids, "val")
copy_files(test_ids, "test")
copy_files(train_ids, "train")

In [8]:
# Ensure the data was uniquely distributed
train_path = save_path / "train"
val_path = save_path / "val"
test_path = save_path / "test"

train_files = OrderedSet([p.name for p in train_path.rglob("*.*")])
val_files = OrderedSet([p.name for p in val_path.rglob("*.*")])
test_files = OrderedSet([p.name for p in test_path.rglob("*.*")])

train_files.intersection(test_files), train_files.intersection(val_files), test_files.intersection(val_files)

(OrderedSet(), OrderedSet(), OrderedSet())